## PROJECT SUMMARY (End‑to‑End Data Pipeline Project)
Goal: Analyze retail sales data to uncover insights about customer behavior, product performance, and revenue trends. Clean, transform, and load this dataset into MySQL, run SQL queries for analysis, and visualize KPIs in Power BI.

## Problem Summary
A retail company is experiencing inconsistent sales performance and limited visibility into customer behavior. Leadership wants to understand:

            Which product categories generate the most revenue

            Which customer segments (age, gender) spend the most

            How monthly sales fluctuate

            Which products drive repeat purchases

            How pricing affects total sales

## TOOLS USED

Programming & Data Processing
Python (Pandas, NumPy)

Jupyter Notebook

Database & Querying
MySQL

SQL (joins, CTEs, window functions, aggregations)

Visualization
Power BI

*Charts, KPIs, slicers, filters

Version Control
Git & GitHub (for project documentation and code)

Automation scripts (Python) --> As needed

## DATASET 
Retail Sales & Customer Analytics Dataset

In [10]:
 #import pandas library to load, clean, transform, and analyze tabular data
import pandas as pd
import numpy as np

## Load and inspect the raw CSV

In [3]:
# Uses Pandas to load CSV file into a DataFrame
data = pd.read_csv("retail_sales_dataset.csv")
df = pd.DataFrame(data)
df

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100
...,...,...,...,...,...,...,...,...,...
995,996,2023-05-16,CUST996,Male,62,Clothing,1,50,50
996,997,2023-11-17,CUST997,Male,52,Beauty,3,30,90
997,998,2023-10-29,CUST998,Female,23,Beauty,4,25,100
998,999,2023-12-05,CUST999,Female,36,Electronics,3,50,150


## Inspect data quality and types

In [4]:
# See basic structure: rows, columns, dtypes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    1000 non-null   int64
 1   Date              1000 non-null   str  
 2   Customer ID       1000 non-null   str  
 3   Gender            1000 non-null   str  
 4   Age               1000 non-null   int64
 5   Product Category  1000 non-null   str  
 6   Quantity          1000 non-null   int64
 7   Price per Unit    1000 non-null   int64
 8   Total Amount      1000 non-null   int64
dtypes: int64(5), str(4)
memory usage: 70.4 KB


In [5]:
# Quick stats for numeric columns (Age, Quantity, Price, Total Amount)
df.describe()

,Transaction ID,Age,Quantity,Price per Unit,Total Amount
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000
mean,500.500000,41.39200,2.514000,179.890000,456.000000
std,288.819436,13.68143,1.132734,189.681356,559.997632
min,1.000000,18.00000,1.000000,25.000000,25.000000
25%,250.750000,29.00000,1.000000,30.000000,60.000000
50%,500.500000,42.00000,3.000000,50.000000,135.000000
75%,750.250000,53.00000,4.000000,300.000000,900.000000
max,1000.000000,64.00000,4.000000,500.000000,2000.000000


In [6]:
# Check a few random rows
df.sample(5, random_state=42)

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
521,522,2023-01-01,CUST522,Male,46,Beauty,3,500,1500
737,738,2023-04-25,CUST738,Male,41,Clothing,2,50,100
740,741,2023-11-30,CUST741,Male,48,Clothing,1,300,300
660,661,2023-07-16,CUST661,Female,44,Clothing,4,25,100
411,412,2023-09-16,CUST412,Female,19,Electronics,4,500,2000


In [7]:
# Look at unique product categories
df['Product Category'].value_counts()

Product Category
Clothing       351
Electronics    342
Beauty         307
Name: count, dtype: int64

In [8]:
# Check how many missing values per column
df.isna().sum()

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

## Clean and Fix the Date Column

In [11]:
# Step 3: Clean and fix the Date column

# 1. Replace Excel overflow values (#######) with NaN
df['Date'] = df['Date'].replace('#######', np.nan)

# 2. Convert the Date column to datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# 3. Check how many dates failed to convert
df['Date'].isna().sum()

# 4. Display a few rows to confirm the fix
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


## Create new useful columns (Feature Engineering)

In [12]:
# Step 4: Feature Engineering

# 1. Extract month (YYYY-MM format)
df['Month'] = df['Date'].dt.to_period('M').astype(str)

# 2. Create age groups
df['Age Group'] = pd.cut(
    df['Age'],
    bins=[17, 25, 35, 45, 60, 100],
    labels=['18-25', '26-35', '36-45', '46-60', '60+']
)

# 3. Preview the new columns
df[['Date', 'Month', 'Age', 'Age Group']].head()

,Date,Month,Age,Age Group
0,2023-11-24,2023-11,34,26-35
1,2023-02-27,2023-02,26,26-35
2,2023-01-13,2023-01,50,46-60
3,2023-05-21,2023-05,37,36-45
4,2023-05-06,2023-05,30,26-35


In [13]:

df = df.rename(columns={
    'Transaction ID': 'transaction_id',
    'Date': 'date',
    'Customer ID': 'customer_id',
    'Gender': 'gender',
    'Age': 'age',
    'Product Category': 'product_category',
    'Quantity': 'quantity',
    'Price per Unit': 'price_per_unit',
    'Total Amount': 'total_amount',
    'Month': 'month',
    'Age Group': 'age_group'
})
df.columns


Index(['transaction_id', 'date', 'customer_id', 'gender', 'age',
       'product_category', 'quantity', 'price_per_unit', 'total_amount',
       'month', 'age_group'],
      dtype='str')

In [14]:
customers_df = df[['customer_id', 'gender', 'age', 'age_group']].drop_duplicates()
customers_df.head()



,customer_id,gender,age,age_group
0,CUST001,Male,34,26-35
1,CUST002,Female,26,26-35
2,CUST003,Male,50,46-60
3,CUST004,Male,37,36-45
4,CUST005,Male,30,26-35


In [15]:
products_df = df[['product_category', 'price_per_unit']].drop_duplicates()
products_df.head()


,product_category,price_per_unit
0,Beauty,50
1,Clothing,500
2,Electronics,30
5,Beauty,30
6,Clothing,25


In [16]:
sales_df = df[['transaction_id', 'date', 'customer_id', 'product_category',
               'quantity', 'total_amount', 'month']].copy()
sales_df.head()


,transaction_id,date,customer_id,product_category,quantity,total_amount,month
0,1,2023-11-24,CUST001,Beauty,3,150,2023-11
1,2,2023-02-27,CUST002,Clothing,2,1000,2023-02
2,3,2023-01-13,CUST003,Electronics,1,30,2023-01
3,4,2023-05-21,CUST004,Clothing,1,500,2023-05
4,5,2023-05-06,CUST005,Beauty,2,100,2023-05


## Design SQL Table Schema (Before Loading Into MySQL)


create three tables:

1. customers
Stores demographic information.
-- --------------------------------------------------------------
Column	                              Type
-- --------------------------------------------------------------
customer_id	                      VARCHAR(20) PRIMARY KEY
gender	                          VARCHAR(10)
age	                              INT
age_group                         VARCHAR(10)

2. products
Stores product category and pricing.
-- ------------------------------------------------------------
Column	                                 Type
-- ------------------------------------------------------------
product_category	               VARCHAR(50) PRIMARY KEY
price_per_unit	                   INT

3. sales
Stores each transaction.
-- --------------------------------------------------------------
Column	                            Type
-- --------------------------------------------------------------
transaction_id              	INT PRIMARY KEY
date	                        DATE
customer_id	                    VARCHAR(20)
product_category	            VARCHAR(50)
quantity	                    INT
total_amount                	INT
month	                        VARCHAR(7)

## Create the MySQL connection in Jupyter

In [18]:
import mysql.connector                              # import python package 

conn = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="password",
    port=3306,
    database="retail_analytics"   # <-- THIS IS THE KEY
)
cursor = conn.cursor()

if conn.is_connected():                             # check if it' connected
    print("Connected... to MySQL")


Connected... to MySQL


In [21]:
cursor.execute("SELECT DATABASE()")
print(cursor.fetchone())


('retail_analytics',)
